# 03 Exogenous features

Price and promotion data are the biggest lever we have beyond sales history, and the easiest place to leak the future. This notebook is about what the model is allowed to know at the cutoff.

Every model gets an information tier, and tiers are never mixed in one ranking:

- **historical_only**: only data observed on or before the cutoff
- **planned_scenario**: also a promotion plan for the next few weeks. This dataset has no real plan, so we stand in realized promotion flags for the first `PLAN_WEEKS` weeks. That is an assumption, not a measured capability.
- **oracle**: realized promotions for the whole horizon
- **leaky_diagnostic**: realized future selling prices, which we show reveal the sales they are meant to predict

The optional notebook 03b covers promotion events and spike adjustment. This notebook runs in about a minute and a half; the optional CatBoost and categorical sections at the end add about nine minutes.


In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler

from shared.prepare_data import load_panel, H, FREQ, ALL_CUTOFFS, TEST_CUTOFF
from shared.features import LAGS, LAG_TRANSFORMS, DATE_FEATURES, LGB_PARAMS, future_promo, promo_features
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard

panel = load_panel()
panel["promo"] = panel.is_promotional.astype(int)
panel["rel_price"] = (panel.sale_price / panel.regular_price.where(panel.regular_price > 0)).fillna(1).clip(0, 3)
panel["discount"] = (1 - panel.rel_price).clip(lower=0)
actuals = panel[["unique_id", "ds", "y"]]
flags = panel[["unique_id", "ds", "promo"]]

## Using this notebook live (30 minutes)

| minutes | section | how |
|---|---|---|
| 8 | 1. Exogenous variables in MLForecast | run; discuss the realized-price check |
| 5 | 2. Lagged drivers | show the feature code |
| 7 | 3. Promotion flags | run the scenario table and the two `X_df` cells |
| 2 | 4. Train at every origin | run (about 1.5 minutes) while discussing section 3 |
| 8 | 5. Results | run and read |

The CatBoost and categorical sections at the end are optional. The categorical section belongs with notebook 04 in the workshop sequence.

## 1. Exogenous variables in MLForecast

### Static, dynamic, historical and future-known

MLForecast treats every extra column in the training frame as a feature. A column that changes over time is **dynamic**, and the model needs its value for every future week, supplied through `X_df` in `predict`. A column that never changes for a series is **static** and goes in `static_features` at fit time. By default MLForecast assumes every extra column is static, so for time-varying drivers we pass `static_features=[]`.

Dynamic inputs are either **historical only** (known up to the cutoff, such as last week's promotion) or **future-known** (known for the target week when we forecast, such as a committed promotion calendar). For every column we ask: what would we put in `X_df` on the cutoff date?

### Why realized future price is not an input

`sale_price` is a realized average selling price. When a product sells nothing in a week the price defaults to the regular price, so the price is partly an outcome of the sales we want to forecast.

In [3]:
at_regular = pd.Series(np.isclose(panel.rel_price, 1), index=panel.index)
pd.DataFrame({"weeks": at_regular.groupby(at_regular).size(), "share_with_zero_sales": (panel.y == 0).groupby(at_regular).mean().round(3)}).rename_axis("price_equals_regular")

,weeks,share_with_zero_sales
price_equals_regular,,
False,171449,0.022
True,41851,0.841


A forecast that is handed next month's realized price has been told which weeks will be zero. We keep one such model below, labeled `leaky_diagnostic`, to see what it does to the forecast. Nothing else in the notebook uses realized future price. The only driver we take from the future is the promotion flag.

## 2. Lab: lagged drivers (B)

With a 14-week recursive forecast the target week can be 14 weeks past the cutoff, so last week's promotion flag is known only for the first forecast week. A lag of at least `H` weeks is known for every target week. B uses promotion and relative price lagged by 14 weeks, plus the promotion share in the 26 weeks ending then. These come from observed history, so `X_df` needs no assumption about the future.

In [4]:
hist_features = panel[["unique_id", "ds", "promo", "rel_price"]].copy()
by_series = hist_features.groupby("unique_id")
hist_features["promo_lag14"] = by_series.promo.shift(H)
hist_features["rel_price_lag14"] = by_series.rel_price.shift(H)
hist_features["promo_freq_26_lag14"] = by_series.promo.transform(lambda s: s.shift(H).rolling(26, min_periods=1).mean())
hist_features = hist_features[["unique_id", "ds", "promo_lag14", "rel_price_lag14", "promo_freq_26_lag14"]].fillna(-1)

## 3. Lab: promotion flags with and without future information (C and E)

One model per feature set is trained on realized history. The scenarios differ only in the promotion flags we put in `X_df` for the forecast weeks:

| scenario | tier | flags for the 14 forecast weeks |
|---|---|---|
| none | historical_only | no promotions |
| planned | planned_scenario | realized flag for the first `PLAN_WEEKS` weeks, then none |
| oracle | oracle | realized flag for all 14 weeks |

E adds features built from the promotion calendar: promotion one and two weeks ago, weeks since the last promotion (capped at 26), the share of the last 13 weeks on promotion, and promotion times weeks-since. At week t each uses flags up to t only; for forecast weeks the flags come from the scenario. See `promo_features` in `shared/features.py`.

In [5]:
PLAN_WEEKS = 4
scenarios = {"none": ("historical_only", 0), "planned": ("planned_scenario", PLAN_WEEKS), "oracle": ("oracle", H)}

basic_cols = ["promo"]
calendar_cols = ["promo", "promo_lag1", "promo_lag2", "promo_freq_13", "weeks_since_promo", "promo_x_gap"]
leaky_cols = ["promo", "rel_price", "discount"]

The scenarios differ only in what goes into `X_df`. Share of series on promotion in each forecast week from the final cutoff:

In [6]:
pd.DataFrame({scenario: future_promo(flags, TEST_CUTOFF, plan_weeks).groupby("ds").promo.mean() for scenario, (_, plan_weeks) in scenarios.items()}).round(3)

,none,planned,oracle
ds,,,
2025-04-16,0.0,0.259,0.259
2025-04-23,0.0,0.260,0.260
2025-04-30,0.0,0.248,0.248
2025-05-07,0.0,0.241,0.241
2025-05-14,0.0,0.000,0.246
2025-05-21,0.0,0.000,0.222
2025-05-28,0.0,0.000,0.244
2025-06-04,0.0,0.000,0.240
2025-06-11,0.0,0.000,0.252


The calendar model's `X_df` for the planned scenario: the flags, and features computed from them week by week.

In [7]:
planned_frame = promo_features(pd.concat([flags[flags.ds <= TEST_CUTOFF], future_promo(flags, TEST_CUTOFF, PLAN_WEEKS)]))
planned_frame.loc[planned_frame.ds > TEST_CUTOFF, ["unique_id", "ds"] + calendar_cols].head(6)

,unique_id,ds,promo,promo_lag1,promo_lag2,promo_freq_13,weeks_since_promo,promo_x_gap
93,10006_S1,2025-04-16,0,0.0,0.0,0.153846,5.0,0.0
94,10006_S1,2025-04-23,1,0.0,0.0,0.153846,6.0,6.0
95,10006_S1,2025-04-30,0,1.0,0.0,0.230769,1.0,0.0
96,10006_S1,2025-05-07,0,0.0,1.0,0.230769,2.0,0.0
97,10006_S1,2025-05-14,0,0.0,0.0,0.153846,3.0,0.0
98,10006_S1,2025-05-21,0,0.0,0.0,0.153846,4.0,0.0


## 4. Train and forecast at every origin

Each model is refit at each cutoff on data up to that cutoff, with the 02 settings and per-series scaler.

In [8]:
def make_model(name):
    return MLForecast(
        models={name: lgb.LGBMRegressor(**LGB_PARAMS)},
        freq=FREQ,
        lags=LAGS,
        lag_transforms=LAG_TRANSFORMS,
        date_features=DATE_FEATURES,
        target_transforms=[LocalStandardScaler()],
        num_threads=1,
    )

mlf_hist = make_model("lgbm_hist_exog")
mlf_promo = make_model("lgbm_promo")
mlf_calendar = make_model("lgbm_promo_calendar")
mlf_leaky = make_model("lgbm_price_leaky")

forecasts = []

for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    in_horizon = lambda d: (d.ds > cutoff) & (d.ds <= cutoff + pd.Timedelta(weeks=H))

    mlf_hist.fit(history.merge(hist_features, on=["unique_id", "ds"]), static_features=[])
    preds = mlf_hist.predict(H, X_df=hist_features[in_horizon(hist_features)])
    forecasts.append(to_long(preds, cutoff, "03_exogenous", "historical_only"))

    for scenario, (tier, plan_weeks) in scenarios.items():
        future = future_promo(flags, cutoff, plan_weeks)
        frame = promo_features(pd.concat([flags[flags.ds <= cutoff], future]))
        past, future = frame[frame.ds <= cutoff], frame[frame.ds > cutoff]

        mlf_promo.fit(history.merge(past[["unique_id", "ds"] + basic_cols], on=["unique_id", "ds"]), static_features=[])
        preds = mlf_promo.predict(H, X_df=future[["unique_id", "ds"] + basic_cols])
        forecasts.append(to_long(preds.rename(columns={"lgbm_promo": f"lgbm_promo_{scenario}"}), cutoff, "03_exogenous", tier))

        mlf_calendar.fit(history.merge(past[["unique_id", "ds"] + calendar_cols], on=["unique_id", "ds"]), static_features=[])
        preds = mlf_calendar.predict(H, X_df=future[["unique_id", "ds"] + calendar_cols])
        forecasts.append(to_long(preds.rename(columns={"lgbm_promo_calendar": f"lgbm_promo_calendar_{scenario}"}), cutoff, "03_exogenous", tier))

    # leaky diagnostic: realized future price and promotion
    mlf_leaky.fit(history.merge(panel[["unique_id", "ds"] + leaky_cols], on=["unique_id", "ds"]), static_features=[])
    preds = mlf_leaky.predict(H, X_df=panel.loc[in_horizon(panel), ["unique_id", "ds"] + leaky_cols])
    forecasts.append(to_long(preds, cutoff, "03_exogenous", "leaky_diagnostic"))

forecasts = pd.concat(forecasts)

## 5. Results

One table, ordered by information tier and then WAPE, on rows every model forecasts. Compare models within a tier; the tiers above it use more information. Development origins first. The final 14 weeks were inspected while building these notebooks, so they are an exploratory benchmark.

In [9]:
reference = load_forecasts(["02_ml_baseline"]).query("model == 'lgbm_baseline'")
columns = ["model", "coverage", "wape", "bias", "mean_rmsse", "median_rmsse", "apc", "apc_median"]

candidates = pd.concat([reference, forecasts])
leaderboard(candidates, actuals, scope="dev")[columns + ["information_tier"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,information_tier
0,lgbm_promo_calendar_none,1.0,0.396,-0.062,1.759,0.860,0.155,0.180,historical_only
1,lgbm_promo_none,1.0,0.396,-0.019,1.739,0.848,0.152,0.177,historical_only
2,lgbm_hist_exog,1.0,0.444,0.141,1.877,0.922,0.134,0.146,historical_only
3,lgbm_baseline,1.0,0.449,0.139,1.865,0.921,0.135,0.145,historical_only
4,lgbm_promo_calendar_planned,1.0,0.381,-0.030,1.737,0.841,0.220,0.233,planned_scenario
5,lgbm_promo_planned,1.0,0.403,0.051,1.745,0.856,0.168,0.192,planned_scenario
6,lgbm_promo_calendar_oracle,1.0,0.352,0.077,1.671,0.794,0.136,0.164,oracle
7,lgbm_promo_oracle,1.0,0.415,0.143,1.746,0.858,0.132,0.153,oracle
8,lgbm_price_leaky,1.0,0.415,0.224,1.662,0.809,0.128,0.144,leaky_diagnostic


In [10]:
leaderboard(candidates, actuals, scope="test")[columns + ["information_tier"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,information_tier
0,lgbm_promo_calendar_none,1.0,0.367,-0.101,1.283,0.687,0.107,0.116,historical_only
1,lgbm_promo_none,1.0,0.374,-0.047,1.272,0.681,0.100,0.106,historical_only
2,lgbm_hist_exog,1.0,0.418,0.119,1.297,0.729,0.100,0.091,historical_only
3,lgbm_baseline,1.0,0.419,0.109,1.302,0.733,0.097,0.091,historical_only
4,lgbm_promo_calendar_planned,1.0,0.360,-0.060,1.265,0.667,0.188,0.181,planned_scenario
5,lgbm_promo_planned,1.0,0.384,0.025,1.268,0.678,0.133,0.136,planned_scenario
6,lgbm_promo_calendar_oracle,1.0,0.322,0.036,1.204,0.618,0.090,0.096,oracle
7,lgbm_promo_oracle,1.0,0.395,0.113,1.263,0.681,0.094,0.094,oracle
8,lgbm_price_leaky,1.0,0.416,0.237,1.235,0.654,0.090,0.088,leaky_diagnostic


### Reading the result

Development origins:

- Lagged drivers (B) are a weak signal: WAPE 0.444 against 0.449 for the baseline.
- Promotion flags help even when every future flag is zero (`none`, historical_only): 0.396, with bias falling from +0.139 to -0.019. A likely reason: with the flag in training, the model can attribute promotion weeks to the promotion rather than to a higher level, so the forecasts stop carrying past spikes forward. The calendar features reach the same WAPE with a more negative bias.
- A four-week plan helps only with the calendar features: 0.381. With the plain flag it is 0.403, worse than no plan.
- All 14 weeks of promotions (oracle) give 0.352 with the calendar features. That is the most promotion information can add here.
- Realized future price (leaky) gives no WAPE gain over the oracle promotion models (0.415) and overforecasts by 22%.

The final window gives the same order for the calendar models: 0.367 (none), 0.360 (planned), 0.322 (oracle).

Try it later: set `PLAN_WEEKS` to 8 or 14 and see how far the planned scenario moves toward the oracle.

In [11]:
save_forecasts(forecasts, "03_exogenous")

## Optional: CatBoost

Three CatBoost candidates, each adding one thing to the one before:

| model | features |
|---|---|
| `catboost_baseline` | the 02 features: sales lags, rolling means, month and Fourier terms |
| `catboost_hist_exog` | plus the lagged drivers of `lgbm_hist_exog` (B above) |
| `catboost_exog_categorical` | plus department, class and subclass as native categorical features |

`lgbm_exog_categorical` has the same features as the last one, with the attributes as pandas `category` columns so LightGBM splits on them as categories. Each feature set then exists for both algorithms, which separates what the features add from what the algorithm adds.

All four are historical_only. The attributes are fixed item properties, known at every origin. The hierarchy ids are numbers in the data but they are labels, so they enter as strings (CatBoost) or categories (LightGBM), never as numeric values. Brand is left out: 701 brands across 2,000 series is close to an item identifier.

CatBoost settings mirror `LGB_PARAMS` where the libraries have equivalents: 300 trees, learning rate 0.05, 80% of features per tree. The target scaler, lags and origins are the same as the LightGBM models. The categorical CatBoost model takes about 35 seconds per fit, so this section runs for about eight minutes.

In [12]:
import time
from catboost import CatBoostRegressor

CB_PARAMS = dict(iterations=300, learning_rate=0.05, depth=6, rsm=0.8, random_seed=0, verbose=0)
hist_cols = ["promo_lag14", "rel_price_lag14", "promo_freq_26_lag14"]
cat_cols = ["department", "class", "subclass"]

items = panel.drop_duplicates("unique_id").set_index("unique_id")
attributes = pd.DataFrame({col: items[f"{col}_id"].astype("Int64").astype("string").fillna("Unknown") for col in cat_cols})
attributes = attributes.rename_axis("unique_id").reset_index()
attributes.head(3)

,unique_id,department,class,subclass
0,10006_S1,108,117,1394
1,10008_S1,27,383,1402
2,10010_S1,12,226,1022


CatBoost's `cat_features` goes to `fit`, through MLForecast's `models_fit_kwargs`. Passing it to the `CatBoostRegressor` constructor fails, because MLForecast copies each model with scikit-learn's `clone`, which can't reproduce that argument. The attributes are static, so they are listed in `static_features` and MLForecast repeats them for the forecast weeks; only the lagged drivers go in `X_df`.

In [13]:
def catboost_model(name):
    return MLForecast(
        models={name: CatBoostRegressor(**CB_PARAMS)},
        freq=FREQ,
        lags=LAGS,
        lag_transforms=LAG_TRANSFORMS,
        date_features=DATE_FEATURES,
        target_transforms=[LocalStandardScaler()],
        num_threads=1,
    )

cb_baseline = catboost_model("catboost_baseline")
cb_hist_exog = catboost_model("catboost_hist_exog")
cb_categorical = catboost_model("catboost_exog_categorical")
lgbm_categorical = make_model("lgbm_exog_categorical")

boost_forecasts, fit_seconds = [], {}
for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    with_exog = history.merge(hist_features, on=["unique_id", "ds"])
    with_categories = with_exog.merge(attributes, on="unique_id")
    future_exog = hist_features[(hist_features.ds > cutoff) & (hist_features.ds <= cutoff + pd.Timedelta(weeks=H))]

    runs = [
        ("catboost_baseline", cb_baseline, history, {}, None),
        ("catboost_hist_exog", cb_hist_exog, with_exog, dict(static_features=[]), future_exog),
        ("catboost_exog_categorical", cb_categorical, with_categories,
         dict(static_features=cat_cols, models_fit_kwargs={"catboost_exog_categorical": {"cat_features": cat_cols}}), future_exog),
        ("lgbm_exog_categorical", lgbm_categorical, with_categories.astype({col: "category" for col in cat_cols}), dict(static_features=cat_cols), future_exog),
    ]
    for name, mlf, train, fit_args, X_df in runs:
        start = time.perf_counter()
        mlf.fit(train, **fit_args)
        boost_forecasts.append(to_long(mlf.predict(H, X_df=X_df), cutoff, "03_catboost"))
        fit_seconds[name] = fit_seconds.get(name, 0) + time.perf_counter() - start

boost_forecasts = pd.concat(boost_forecasts)

### Are the attributes treated as categories?

The features each fitted model sees, and how it treats them.

In [14]:
catboost_fit = cb_categorical.models_["catboost_exog_categorical"]
lgbm_fit = lgbm_categorical.models_["lgbm_exog_categorical"]
lgbm_infos = lgbm_fit.booster_.dump_model()["feature_infos"]
pd.DataFrame({
    "catboost": ["categorical" if i in catboost_fit.get_cat_feature_indices() else "numeric" for i in range(len(catboost_fit.feature_names_))],
    "lightgbm": ["categorical" if lgbm_infos[f]["values"] else "numeric" for f in catboost_fit.feature_names_],
}, index=catboost_fit.feature_names_)

,catboost,lightgbm
promo_lag14,numeric,numeric
rel_price_lag14,numeric,numeric
promo_freq_26_lag14,numeric,numeric
department,categorical,categorical
class,categorical,categorical
subclass,categorical,categorical
lag1,numeric,numeric
lag2,numeric,numeric
lag3,numeric,numeric
lag4,numeric,numeric


### Results

With `lgbm_baseline` and `lgbm_hist_exog`, the LightGBM models with the first two feature sets. Fit time is fit plus 14-week forecast, summed over the ten origins; a LightGBM baseline fit takes under a second (notebook 05).

In [15]:
boost_candidates = pd.concat([reference, forecasts[forecasts.model == "lgbm_hist_exog"], boost_forecasts])
pd.Series(fit_seconds, name="fit_seconds_all_origins").round(1)

catboost_baseline             21.7
catboost_hist_exog            20.4
catboost_exog_categorical    405.5
lgbm_exog_categorical         19.4
Name: fit_seconds_all_origins, dtype: float64

In [16]:
leaderboard(boost_candidates, actuals, scope="dev")[columns + ["coverage"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,lgbm_exog_categorical,1.0,0.422,0.099,1.850,0.891,0.157,0.156,1.0
1,catboost_exog_categorical,1.0,0.434,0.129,1.810,0.898,0.118,0.131,1.0
2,catboost_hist_exog,1.0,0.438,0.131,1.818,0.908,0.118,0.133,1.0
3,catboost_baseline,1.0,0.442,0.130,1.815,0.910,0.126,0.141,1.0
4,lgbm_hist_exog,1.0,0.444,0.141,1.877,0.922,0.136,0.151,1.0
5,lgbm_baseline,1.0,0.449,0.139,1.865,0.921,0.136,0.149,1.0


In [17]:
leaderboard(boost_candidates, actuals, scope="test")[columns + ["coverage"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,lgbm_exog_categorical,1.0,0.396,0.105,1.266,0.706,0.113,0.098,1.0
1,lgbm_hist_exog,1.0,0.418,0.119,1.297,0.729,0.102,0.091,1.0
2,lgbm_baseline,1.0,0.419,0.109,1.302,0.733,0.101,0.091,1.0
3,catboost_baseline,1.0,0.424,0.113,1.312,0.746,0.092,0.092,1.0
4,catboost_exog_categorical,1.0,0.425,0.128,1.305,0.736,0.093,0.084,1.0
5,catboost_hist_exog,1.0,0.427,0.126,1.307,0.741,0.092,0.088,1.0


The same WAPE arranged by feature set and algorithm. Moving down a column changes the features; moving across a row changes the algorithm.

In [18]:
grid = pd.DataFrame({
    "lightgbm": ["lgbm_baseline", "lgbm_hist_exog", "lgbm_exog_categorical"],
    "catboost": ["catboost_baseline", "catboost_hist_exog", "catboost_exog_categorical"],
}, index=["basic", "+ lagged drivers", "+ categorical attributes"])
dev_wape = leaderboard(boost_candidates, actuals, scope="dev").set_index("model").wape
test_wape = leaderboard(boost_candidates, actuals, scope="test").set_index("model").wape
pd.concat({"development": grid.apply(lambda col: col.map(dev_wape)), "final 14 weeks": grid.apply(lambda col: col.map(test_wape))}, axis=1).round(3)

development          final 14 weeks         
                            lightgbm catboost       lightgbm catboost
basic                          0.449    0.442          0.419    0.424
+ lagged drivers               0.444    0.438          0.418    0.427
+ categorical attributes       0.422    0.434          0.396    0.425

### Reading the result

- **Algorithm, same features.** CatBoost is slightly better than LightGBM on the development origins with the basic and lagged-driver features (0.442 against 0.449, 0.438 against 0.444), and slightly worse in the final window (0.424 against 0.419, 0.427 against 0.418). Neither difference holds across both periods.
- **Categorical attributes.** They help LightGBM much more than CatBoost: development WAPE 0.444 to 0.422 for LightGBM, 0.438 to 0.434 for CatBoost; final window 0.418 to 0.396 against 0.427 to 0.425. `lgbm_exog_categorical` is the best model in this table in both periods.
- **Bias and stability.** Every model here overforecasts (bias +0.10 to +0.14), as the L2 baseline does. CatBoost revises less between origins (development APC 0.118 to 0.126, against 0.136 to 0.157) and has a slightly lower development mean RMSSE.
- **Cost.** CatBoost takes about 2 seconds per fit without categorical features and about 34 seconds with them. LightGBM with the same categorical features takes about 1 second.

The gain from the hierarchy attributes is mostly a feature effect that LightGBM uses better here, not a reason to switch algorithm. None of these models reaches the best historical-only models from notebooks 05 to 07 (WAPE 0.364 to 0.369 in the final window), which change the objective or horizon strategy rather than the features.

Try it later: give the categorical models the L1 objective from notebook 06 (`loss_function="MAE"` for CatBoost, `objective="l1"` for LightGBM).

In [19]:
save_forecasts(boost_forecasts, "03_catboost")

## Optional: categorical attributes on the strongest LightGBM models

Department, class and subclass helped the basic LightGBM by about 2 WAPE points. Does that carry over to the two strongest historical-only LightGBM models?

| parent | new candidate |
|---|---|
| `lgbm_l1_std` (notebook 06: L1 objective, standardized target) | `lgbm_l1_std_categorical` |
| `lgbm_promo_calendar_none` (above: promotion-calendar features, no future promotions) | `lgbm_promo_calendar_categorical` |

Each new model is its parent plus the three attributes as pandas `category` columns in `static_features`. Objective, scaler, lags, calendar features, training windows and origins are unchanged. The parents are refit here with the same code so that fit times are measured the same way; their forecasts are the saved ones.

In [20]:
from shared.evaluation import score
from shared.prepare_data import DEV_CUTOFFS


def l1_model(name):
    return MLForecast(
        models={name: lgb.LGBMRegressor(**dict(LGB_PARAMS, objective="l1"))},
        freq=FREQ,
        lags=LAGS,
        lag_transforms=LAG_TRANSFORMS,
        date_features=DATE_FEATURES,
        target_transforms=[LocalStandardScaler()],
        num_threads=1,
    )

as_category = {col: "category" for col in cat_cols}
strong_forecasts, strong_seconds = [], {}

for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    frame = promo_features(pd.concat([flags[flags.ds <= cutoff], future_promo(flags, cutoff, 0)]))
    past, future = frame[frame.ds <= cutoff], frame[frame.ds > cutoff]
    calendar_train = history.merge(past[["unique_id", "ds"] + calendar_cols], on=["unique_id", "ds"])
    calendar_future = future[["unique_id", "ds"] + calendar_cols]

    runs = [
        ("lgbm_l1_std", l1_model("lgbm_l1_std"), history, {}, None),
        ("lgbm_l1_std_categorical", l1_model("lgbm_l1_std_categorical"),
         history.merge(attributes, on="unique_id").astype(as_category), dict(static_features=cat_cols), None),
        ("lgbm_promo_calendar_none", make_model("lgbm_promo_calendar_none"), calendar_train, dict(static_features=[]), calendar_future),
        ("lgbm_promo_calendar_categorical", make_model("lgbm_promo_calendar_categorical"),
         calendar_train.merge(attributes, on="unique_id").astype(as_category), dict(static_features=cat_cols), calendar_future),
    ]
    for name, mlf, train, fit_args, X_df in runs:
        start = time.perf_counter()
        mlf.fit(train, **fit_args)
        strong_forecasts.append(to_long(mlf.predict(H, X_df=X_df), cutoff, "03_categorical"))
        strong_seconds[name] = strong_seconds.get(name, 0) + time.perf_counter() - start

strong_forecasts = pd.concat(strong_forecasts)
strong_forecasts = strong_forecasts[strong_forecasts.model.str.endswith("_categorical")]

### Each new model against its parent

The parents come from the saved forecasts of notebooks 03 and 06. Fit time is fit plus 14-week forecast, summed over the ten origins.

In [21]:
parents = pd.concat([forecasts[forecasts.model == "lgbm_promo_calendar_none"], load_forecasts(["06_objectives"]).query("model == 'lgbm_l1_std'")])
strong_candidates = pd.concat([parents, strong_forecasts])
pd.Series(strong_seconds, name="fit_seconds_all_origins").round(1)

lgbm_l1_std                         6.6
lgbm_l1_std_categorical            11.1
lgbm_promo_calendar_none            6.6
lgbm_promo_calendar_categorical    10.2
Name: fit_seconds_all_origins, dtype: float64

In [22]:
leaderboard(strong_candidates, actuals, scope="dev")[columns + ["coverage"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,lgbm_l1_std_categorical,1.0,0.359,-0.072,1.728,0.830,0.142,0.162,1.0
1,lgbm_l1_std,1.0,0.378,-0.039,1.705,0.833,0.156,0.194,1.0
2,lgbm_promo_calendar_categorical,1.0,0.394,-0.083,1.809,0.863,0.165,0.182,1.0
3,lgbm_promo_calendar_none,1.0,0.396,-0.062,1.759,0.860,0.153,0.184,1.0


In [23]:
leaderboard(strong_candidates, actuals, scope="test")[columns + ["coverage"]].round(3)

,model,coverage,wape,bias,mean_rmsse,median_rmsse,apc,apc_median,coverage
0,lgbm_l1_std_categorical,1.0,0.350,-0.020,1.257,0.667,0.117,0.113,1.0
1,lgbm_promo_calendar_categorical,1.0,0.361,-0.089,1.260,0.682,0.108,0.109,1.0
2,lgbm_promo_calendar_none,1.0,0.367,-0.101,1.283,0.687,0.107,0.114,1.0
3,lgbm_l1_std,1.0,0.369,-0.003,1.265,0.671,0.123,0.136,1.0


WAPE at every origin. The last column counts the origins where the categorical model beats its parent.

In [24]:
strong_wape = pd.concat([score(strong_candidates[strong_candidates.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in list(DEV_CUTOFFS) + [TEST_CUTOFF]])
strong_wape = strong_wape.pivot(index="model", columns="cutoff", values="wape")
pairs = {"lgbm_l1_std_categorical": "lgbm_l1_std", "lgbm_promo_calendar_categorical": "lgbm_promo_calendar_none"}
change = pd.DataFrame({new: (strong_wape.loc[new] - strong_wape.loc[parent]) * 100 for new, parent in pairs.items()}).T
change.round(2).assign(better_at=(change < 0).sum(axis=1))

cutoff,2024-07-17 00:00:00,2024-08-14 00:00:00,2024-09-11 00:00:00,2024-10-09 00:00:00,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-04-09 00:00:00,better_at
lgbm_l1_std_categorical,-1.57,-1.10,-0.69,-1.41,-1.64,-4.76,-2.17,-1.84,8
lgbm_promo_calendar_categorical,-1.47,0.43,0.85,0.10,0.10,-0.89,-0.25,-0.61,4


### Reading the result

- **`lgbm_l1_std_categorical` improves on its parent everywhere.** Development WAPE 0.359 against 0.378 and final window 0.350 against 0.369, better at all eight origins (by 0.7 to 4.8 points). It is the best single historical-only model in Day 1.
- **`lgbm_promo_calendar_categorical` barely moves.** Development WAPE 0.394 against 0.396, better at four of eight origins; final window 0.361 against 0.367. The promotion-calendar features may already capture part of what the attributes add.
- **Bias.** Both new models forecast lower than their parents. For the L1 model that pushes the development bias from -0.039 to -0.072 (final window -0.003 to -0.020).
- **RMSSE and stability.** Development mean RMSSE is slightly worse for both (1.728 against 1.705, 1.809 against 1.759), while the final window is slightly better. APC falls for the L1 model (0.142 against 0.156) and rises for the calendar model (0.165 against 0.153).
- **Cost.** The categorical models take about 60% longer to fit: 26 against 16 seconds and 24 against 15 seconds over ten origins.

The idea came from the basic-model result, which was read on the final window too, but the L1 gain also holds at every development origin.

In [25]:
save_forecasts(strong_forecasts, "03_categorical")